In [0]:
%run ../../00_common/data_utils

In [0]:
def handle_job_limit(task_id, config_json):
    """
    按 market 限流，将选中的 Pending 记录标记为 InProgress 并分配 task_id。

    === 限流维度 ===
    以 (BrandCode, New_UniversalKey) 为计数单位，而非行数。
    例如：同一个 Ukey 在 3 个 Brand 下 → 计数为 3，而不是 N 行。
    限流粒度与下游 process_anonymization_data 处理粒度一致。

    === 优先级策略 ===
    每个 market 的待处理维度按以下顺序选取（最多 max_job_proccess_count 个）：
    1. LastActivityTime 升序（先活跃的先处理）
    2. ExportDate 升序（先到期的先处理）
    3. create_time 升序（先创建的先处理）
    4. cal_uuid 升序（确定性 tie-breaker）
    维度选好后，join 回 log 表取齐该维度下所有 ConsumerId 行。

    === 未配置 market ===
    config 中未列出的 market 不限流，全量取走。

    === 副作用 ===
    更新 log 表：status = InProgress, task_id = 当前 task_id
    """
    anonymization_db = get_env_config('silver_mdm_anonymization_database')
    log_table = f"{anonymization_db}.t_consumer_anonymization_log"

    # 解析 job.yaml 传入的 config JSON 数组
    # 格式：[{"market_code": "HKG", "rate_limit_enable": true, "max_job_proccess_count": 500}, ...]
    config_list = json.loads(config_json)

    # =========================================================================
    # Step 1: 加载待处理记录
    # =========================================================================
    # 只取 status=Pending 且 active_type=anonymization 的记录。
    # not_match 类型的记录不需要下游处理（无匹配 consumer），排除以节省限流配额。
    log_df = (
        spark.table(log_table)
        .filter(
            (F.col("status") == ANON_STATUS_PENDING) &
            (F.col("active_type") == ANON_ACTIVE_TYPE_ANONYMIZATION)
        )
    )

    # =========================================================================
    # Step 2: 构建限流维度表 dim_df
    # =========================================================================
    # 每个 (MarketCode, BrandCode, New_UniversalKey) 为一个限流维度，聚合出：
    #   min_last_activity_time — 该维度下所有 ConsumerId 行中最早的 LastActivityTime
    #   min_export_date        — 该维度下所有 ConsumerId 行中最早的 ExportDate
    #   min_create_time        — 该维度下所有 ConsumerId 行中最早的 create_time
    #   min_cal_uuid           — 该维度下最小的 cal_uuid（确定性 tie-breaker）
    #
    # 为什么要 checkpoint？
    #   dim_df 后续会被每个 market 分别 filter + orderBy + limit，触发多次物理计算。
    #   eager checkpoint 将中间结果物化，避免重复扫描 log 表。
    dim_df = (
        log_df
        .groupBy("MarketCode", "BrandCode", "New_UniversalKey")
        .agg(
            F.min("LastActivityTime").alias("min_last_activity_time"),
            F.min("ExportDate").alias("min_export_date"),
            F.min("create_time").alias("min_create_time"),
            F.min("cal_uuid").alias("min_cal_uuid")
        )
        .checkpoint(eager=True)
    )

    # =========================================================================
    # Step 3: 按 MarketCode 汇总统计
    # =========================================================================
    # total_pending — 该 market 下所有 Pending 的去重维度数
    market_stats_rows = (
        dim_df
        .groupBy("MarketCode")
        .agg(F.count(F.lit(1)).alias("total_pending"))
        .collect()
    )
    market_stats = {row["MarketCode"]: row["total_pending"] for row in market_stats_rows}

    # =========================================================================
    # Step 4: 逐 market 选取待处理记录
    # =========================================================================
    selected_dfs = []         # 各 market 选出的 DataFrame，最终 union
    configured_markets = set()  # 记录已配置的 market，用于后续识别未配置 market

    for item in config_list:
        market = item["market_code"]
        configured_markets.add(market)

        rate_limit_enable = item.get("rate_limit_enable", False)
        max_count = item.get("max_job_proccess_count", 0)

        total_pending = market_stats.get(market, 0)

        if total_pending == 0:
            print(f"{market}: no Pending records")
            continue

        if rate_limit_enable and max_count >= 0:
            # --- 限流逻辑 ---
            # 从 dim_df 中选出本 market 下最多 max_count 个 (BrandCode, New_UniversalKey)：
            #   orderBy:
            #     1. min_last_activity_time ASC — 先活跃的先处理
            #     2. min_export_date ASC       — 先到期的先处理
            #     3. min_create_time ASC       — 先创建的先处理
            #     4. min_cal_uuid ASC          — 确定性 tie-breaker
            #   .limit(max_count)             — 维度数上限
            selected_dim = (
                dim_df
                .filter(F.col("MarketCode") == market)
                .orderBy(
                    F.col("min_last_activity_time").asc(),
                    F.col("min_export_date").asc(),
                    F.col("min_create_time").asc(),
                    F.col("min_cal_uuid").asc()
                )
                .limit(max_count)
                .select("BrandCode", "New_UniversalKey")
            )

            # 将选中的维度 join 回 log_df，取齐该维度下所有 ConsumerId 行
            # （一个 (BrandCode, Ukey) 维度下可能有多行，对应不同 SourceSystemCode/ConsumerId）
            selected = (
                log_df
                .filter(F.col("MarketCode") == market)
                .join(selected_dim, ["BrandCode", "New_UniversalKey"], "inner")
            )

            actual_dims = min(total_pending, max_count)
            print(f"{market}: rate_limit enabled, max={max_count} ukey dims, "
                  f"selected={actual_dims} dims")
        else:
            # --- 不限流：该 market 全量取走 ---
            selected = log_df.filter(F.col("MarketCode") == market)
            print(f"{market}: rate_limit disabled, selected {total_pending} ukeys")

        selected_dfs.append(selected)

    # =========================================================================
    # Step 5: 未配置 market 全量取走
    # =========================================================================
    # config 中未列出的 market 不做限流，所有 Pending 记录全部处理。
    other_count = sum(
        cnt for m, cnt in market_stats.items()
        if m not in configured_markets
    )
    if other_count > 0:
        other_markets = log_df.filter(~F.col("MarketCode").isin(list(configured_markets)))
        print(f"other markets (no config): selected {other_count} ukeys")
        selected_dfs.append(other_markets)

    # =========================================================================
    # Step 6: 无记录则提前返回
    # =========================================================================
    if not selected_dfs:
        print("No Pending records to process")
        return

    # =========================================================================
    # Step 7: Union 所有选中记录，更新 status → InProgress
    # =========================================================================
    # 使用 cal_uuid（写入 log 时生成的 UUID）作为 merge key，精准更新选中行。
    selected_df = selected_dfs[0]
    for df in selected_dfs[1:]:
        selected_df = selected_df.union(df)

    # 去重 cal_uuid 并 checkpoint，避免 union 后重复扫描
    update_keys = selected_df.select("cal_uuid").distinct()
    update_keys = update_keys.checkpoint(eager=True)
    total_selected = update_keys.count()
    print(f"Total selected: {total_selected} rows")

    # 原子更新 status + task_id，将选中记录标记为当前任务负责处理
    DeltaTable.forName(spark, log_table).alias("target").merge(
        update_keys.alias("source"),
        "target.cal_uuid = source.cal_uuid"
    ).whenMatchedUpdate(set={
        "status": F.lit(ANON_STATUS_IN_PROGRESS),
        "task_id": F.lit(task_id)
    }).execute()

    print(f"Updated {total_selected} records to InProgress")

In [0]:
task_id = dbutils.widgets.get("task_id")
config = dbutils.widgets.get("config")
print(f"task_id: {task_id}")
print(f"config: {config}")

step_name = "handle_job_limit"
step_num = "01_2"
project = "dataanonymization"
log_table_name = f"{get_env_config('config_database')}.t_task_step_log"

start_time = datetime.now()
status = "SUCCESS"
message = "completed"

try:
    spark.sparkContext.setCheckpointDir(f"{get_env_config('checkpoint_path_consumer_master')}/{task_id}")
    handle_job_limit(task_id, config)
except Exception as e:
    status = "FAILED"
    message = f"{type(e).__name__}: {str(e)}"
    raise
finally:
    end_time = datetime.now()
    append_step_log(
        log_table_name=log_table_name,
        task_id=task_id,
        step_num=step_num,
        step_name=step_name,
        start_time=start_time,
        end_time=end_time,
        status=status,
        message=message,
        project=project
    )
